#Module importieren & Datensatz einlesen

In [ ]:
# Module importieren
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [ ]:
data=pd.read_csv('SPAM text message 20170820 - Data.csv')

In [ ]:
data.head()

#Datenimport und Exploration

In [ ]:
data.groupby('Category').describe().T

In [ ]:
import seaborn as sns
plt.figure(figsize=(4,4))
sns.countplot(data['Category'])

In [ ]:
#Daten ausbalancieren, da sie unausgewogen sind
ham_msg = data[data.Category =='ham']
spam_msg = data[data.Category=='spam']

In [ ]:
#Zufallsentnahme von Daten aus ham_ msg
ham_msg=ham_msg.sample(n=len(spam_msg),random_state=42)


In [ ]:
print(ham_msg.shape,spam_msg.shape)

#Datenvorbereitung

In [ ]:
# Variante

import seaborn as sns
import matplotlib.pyplot as plt

# Zusammenf?hren der ausgew?hlten ham- und spam-Nachrichten
balanced_data = pd.concat([ham_msg, spam_msg]).reset_index(drop=True)

# Plotting
plt.figure(figsize=(4, 4))
sns.countplot(balanced_data['Category'])
plt.title('Distribution of ham and spam email messages (after downsampling)')
plt.xlabel('Message types')
plt.show()



In [ ]:
balanced_data.head()

In [ ]:
balanced_data['label']=balanced_data['Category'].map({'ham':0,'spam':1})

In [ ]:
train_msg, test_msg, train_labels, test_labels =train_test_split(balanced_data['Message'],balanced_data['label'],test_size=0.2,random_state=434)

#Tokenisierung und Padding

In [ ]:
vocab_size=500
oov_tok='<OOV>'
max_len=50

In [ ]:
#Vorverarbeitung, um aus Text Token zu machen
token=Tokenizer(num_words=vocab_size,oov_token=oov_tok)
token.fit_on_texts(train_msg)

In [ ]:
word_index=token.word_index
word_index

In [ ]:
padding_type='post'
truncate_type='post'
Trainning_seq=token.texts_to_sequences(train_msg)
Trainning_pad=pad_sequences(Trainning_seq,maxlen=50,padding=padding_type,truncating=truncate_type)

In [ ]:
Testing_seq=token.texts_to_sequences(test_msg)
Testing_pad=pad_sequences(Testing_seq,maxlen=50,padding=padding_type,truncating=truncate_type)

#Modelldefinition und Training

In [ ]:
#model
model=tf.keras.models.Sequential([tf.keras.layers.Embedding(vocab_size,16,input_length=50),
                                  tf.keras.layers.GlobalAveragePooling1D(),
                                  tf.keras.layers.Dense(32,activation='relu'),
                                  tf.keras.layers.Dropout(0.3),
                                  tf.keras.layers.Dense(1,activation='sigmoid')])


In [ ]:
model.compile(loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),metrics=['accuracy'],optimizer='adam')

In [ ]:
epoch=30
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)
history=model.fit(Trainning_pad, train_labels ,validation_data=(Testing_pad, test_labels),epochs=epoch,callbacks=[early_stop],verbose=2)

#Evaluierung und Visualisierung der Trainingshistorie

In [ ]:
model.evaluate(Testing_pad, test_labels)

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')

#Vorhersage

In [ ]:
predict_msg = ["Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",
          "Thanks for your subscription to Video UK your mobile will be charged ?5/month Please confirm by replying YES or NO",
          "Then we meet at three. I'm looking forward to it. Greetings to grandma"]

In [ ]:
def predict_spam(predict_msg):
    new_seq = token.texts_to_sequences(predict_msg)
    padded = pad_sequences(new_seq, maxlen =50,
                      padding = padding_type,
                      truncating='post')
    return (model.predict(padded))
predict_spam(predict_msg)

#Klassifizierung der Vorhersagen

In [ ]:
# Variante mit Klassifizierung


# Tokenisierung und Padding der neuen Nachrichten
predict_seq = token.texts_to_sequences(predict_msg)
predict_pad = pad_sequences(predict_seq, maxlen=50, padding=padding_type, truncating=truncate_type)

# Vorhersage mit dem trainierten Modell
predictions = model.predict(predict_pad)

# Ausgabe der Vorhersagen
for i, msg in enumerate(predict_msg):
    print(f"Message: {msg}")
    print(f"Predicted Label: {round(predictions[i][0])} (1 for 'spam', 0 for 'ham')")
    print("-----------")


In [ ]:
loss, accuracy = model.evaluate(Testing_pad, test_labels)
print(f'Test Loss: {loss}, Test Accuracy: {accuracy}')

#Zugabe: Spa? mit Auswertungen

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Trainingsdaten vorhersagen
train_predictions = model.predict(Trainning_pad)
train_predictions = (train_predictions > 0.5).astype(int)

# Testdaten vorhersagen
test_predictions = model.predict(Testing_pad)
test_predictions = (test_predictions > 0.5).astype(int)

# Pr?zision, R?ckruf und F1-Score f?r Trainingsdaten
train_precision = precision_score(train_labels, train_predictions)
train_recall = recall_score(train_labels, train_predictions)
train_f1 = f1_score(train_labels, train_predictions)

# Pr?zision, R?ckruf und F1-Score f?r Testdaten
test_precision = precision_score(test_labels, test_predictions)
test_recall = recall_score(test_labels, test_predictions)
test_f1 = f1_score(test_labels, test_predictions)

# Ausgabe der Ergebnisse
print("Trainingsdaten:")
print(f"Pr?zision: {train_precision:.4f}")
print(f"R?ckruf: {train_recall:.4f}")
print(f"F1-Score: {train_f1:.4f}")

print("\nTestdaten:")
print(f"Pr?zision: {test_precision:.4f}")
print(f"R?ckruf: {test_recall:.4f}")
print(f"F1-Score: {test_f1:.4f}")


In [ ]:
model.fit(Trainning_pad, train_labels, epochs=epoch, validation_data=(Testing_pad, test_labels), callbacks=[early_stop])

In [ ]:
loss, accuracy = model.evaluate(Testing_pad, test_labels)
print(f'Test Loss: {loss}, Test Accuracy: {accuracy}')